# B.7 工具選擇怎樣才算可靠？


每題都選 TOOL，在只考計算的題目上也可能拿高分。混合 DIRECT、TOOL、ASK 的題目，才量得到漏用工具、多用工具，以及該問卻直接答。先用六筆人工結果讀這些分母：`gold` 是每題的正確卡，`predicted` 是假設的選卡結果。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
gold = ["TOOL", "TOOL", "DIRECT", "DIRECT", "ASK", "ASK"]
predicted = ["TOOL", "DIRECT", "TOOL", "DIRECT", "ASK", "DIRECT"]
correct = sum(a == b for a, b in zip(gold, predicted, strict=True))
needed = sum(a == "TOOL" for a in gold)
missed = sum(a == "TOOL" and b != "TOOL" for a, b in zip(gold, predicted, strict=True))
not_needed = len(gold) - needed
extra = sum(a != "TOOL" and b == "TOOL" for a, b in zip(gold, predicted, strict=True))
print("動作選對", correct, "/", len(gold))
print("需要工具卻沒選", missed, "/", needed)
print("不需要工具卻選了", extra, "/", not_needed)

`zip(gold, predicted, strict=True)` 把兩份清單按位置配成一對對（`zip` 見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)），`strict=True` 要求兩份一樣長，否則報錯，避免少配一題卻沒發現。`sum(a == b for ...)` 把每題的 True 當 1、False 當 0 加總。動作選對 3/6。需要工具的有 2 題，漏選 1 題，所以是 1/2；不需要工具的有 4 題，多選 1 題，所以是 1/4。後兩項的分母不同，不能都除以 6。選對卡也還沒有真正運算；真正的呼叫與最後答案要另外計分。

選卡實驗把同一對數字的加法、解釋、照抄與工具狀態變化歸成一個家族（[5.12](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)）。測試資料有 6 個家族；每個家族有 4 種任務×2 種原本的問法×計算器可用、停用 2 種狀態＝16 題，共 6×16＝96 題。另有一組新問法：每種任務只換成一種沒訓練過的問法，每家族 4×1×2＝8 題，共 6×8＝48 題。原本的問法配上沒訓練過的數字對，得到 96/96；新問法只有 42/48。總數看起來不錯，但新問法裡 6 道該用工具的題全部錯選成 ASK，是 0/6，例如「幫我把 1 與 2 相加，給我答案。」這個錯例指出問法的泛化不足，不是計算器傳回了錯數字。

另外拿完全相同的問題，只把 system 從計算器可用換成停用：測試時分別生成 TOOL 與 ASK，支持模型在有限的句型內會依狀態選擇。ASK 仍只是一張卡，模型沒有寫出求助的原因；資料裡人寫的「缺數量」不能當成模型已經解釋過。

可靠性要逐類、逐條件說明。不能把選卡模型的 96/96 和另一個 JSON 模型的 19/20 拼成端到端的成功率，也不能用 42/48 蓋過新問法下工具選擇全部失敗。端到端的能力，必須由同一個模型、實際串起來的完整軌跡支持。

練習只把 `predicted` 的第二項 `"DIRECT"` 改成 `"TOOL"`，先預測：動作選對 4/6，需要工具卻沒選 0/2，不需要工具卻選了仍是 1/4；再執行核對。

<details>
<summary>補充：選卡實驗的配方、題型與完整分項</summary>

選卡實驗把同一對數字的加法、照抄、解釋及工具狀態變化，放進同一個家族。家族先分成訓練、驗證與測試資料；同一筆問題不會同時出現在兩邊。訓練資料用來更新參數，驗證資料用來觀察，測試資料只在配方固定後做最後檢查，不按它的結果選步數或重訓。另一組新問法則專門檢查：只換措辭，原本的選擇是否仍成立。題目來自有限的模板，這些仍是受控的自然語言小世界，不能外推成任意問題的工具判斷能力。

數字範圍 0 到 9，1 加 2 與 2 加 1 同一個家族，1 加 1 也保留；測的是沒見過的數字組合，每個單獨的數字仍可能出現在訓練資料的其他家族。

每個家族都有以下四種任務；每個問法再配上計算器可用、停用兩種 system 狀態。原本的問法每家族共 4 種任務×2 個問法×2 種工具狀態＝16 筆，新問法則是 4×1×2＝8 筆。

| 任務 | 原本問法數／新問法數 | 計算器可用時的正確卡 | 計算器停用時的正確卡 |
| --- | --- | --- | --- |
| 求加法的精確數值 | 2／1 | TOOL | ASK |
| 用積木等方式解釋加法 | 2／1 | DIRECT | DIRECT |
| 照抄給出的兩個數字 | 2／1 | DIRECT | DIRECT |
| 報價，尚未提供購買數量 | 2／1 | ASK | ASK |

例如「單價 1 元，運費 2 元，總價多少？」在這份報價流程裡還缺購買數量，標為 ASK。測試資料是含「1 與 2」這一家在內的六個家族；每家族原本的問法 16 筆，共 96 題；新問法 8 筆，共 48 題。

選卡模型從隨機參數開始學示範答案，配方在最後檢查之前就固定。完整的更新設定與原始生成保留在本節末的實驗紀錄，沒有依測試結果重訓或改參數。

| 核對項目 | 原本問法、沒見過的數字對 | 新問法 |
| --- | --- | --- |
| 動作選對 | 96/96 | 42/48 |
| 應使用工具且選中TOOL | 12/12 | 0/6 |
| 不該使用工具卻選TOOL | 0/84 | 0/42 |
| 計算器停用卻選TOOL | 0/48 | 0/24 |
| 永遠選DIRECT的對照 | 48/96 | 24/48 |
| 永遠選TOOL的對照 | 12/96 | 6/48 |

表的第一列與最後兩個對照，都以全部題數為分母。每個家族在原本的問法裡有兩道「計算器可用的加法題」該選 TOOL，六個家族共 12 道；其餘 96−12＝84 道不該選。每家族 16 筆中有一半停用工具，所以停用的分母是 6×8＝48。新問法每家族只有一個加法問法，因此相應的分母變成 6、48−6＝42 與 6×4＝24。這些子集彼此重疊，不能把各列分母相加，當成另一批題目。

例如測試題「1 加 2 等於多少？」在 system 只寫「計算器可用。」時生成 TOOL；把 system 換成「計算器停用。」就生成 ASK。決策規則放在示範答案裡，system 只提供工具狀態，和 [B.6](https://birdhackor.github.io/tiny-perceptron-vlm/B.6.html) 的資料格式相同。同一數字對的解釋與照抄題則生成 DIRECT。這支持有限句型內已學到區分，不能證明模型先估算了自己算術答對的機率。

新問法沿用測試資料的六個家族，但每種任務換一個新模板，各配可用、停用兩種狀態，共 48 題。計算器可用時，「幫我把 1 與 2 相加，給我答案。」卻生成 ASK。六道該用工具的題全部如此；42/48 的總分其實掩蓋了工具選擇在這個新問法上完全失敗。缺資訊的判定也遵守人訂的「報價先取得購買數量」規則，沒有測量模型能否推導零單價這類特例。

ASK 只是這個實驗的一張卡，模型沒有寫出「請補數量」或「請啟用工具」這類完整的說明。資料另外記下求助原因，方便人查標註，不能把那些人寫的原因當成模型的回答。

完整的[實驗紀錄 tool_choice.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/tool_choice.json)保留每題的問題、工具狀態、正確卡與實際生成：測試資料在 `results.test`，新問法在 `results.paraphrase_diagnostic`，兩者的 `metrics` 是上表的各項，`baselines` 是兩種永遠選同一張卡的對照。[實驗程式](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/course_experiments/tool_choice.py)保留資料建立與切分方法。分數只評選卡，不含 JSON 請求的生成、工具運算或最後答案。即使選卡全對，也不能接上 [B.4](https://birdhackor.github.io/tiny-perceptron-vlm/B.4.html) 另一個模型的 19/20，拼成一個沒有實際跑過的端到端成功率。

</details>
